# Welcome to Colab!

In [2]:
pip install requests netmiko napalm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.8/535.8 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 22.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.9 MB/s  0:00:00
  Created wheel for ncclient: filename=ncclient-0.7.0-py3-none-any.whl size=94121 sha256=8e24b31c736b2e29569f5c35bfdd05a6e01b1598947ec329b7b1aaeee9e7eaaf
  Stored in directory: /root/.cache/pip/wheels/7a/4b/65/e346ceab2f8f7921b505e31b2392d14913140aef5c3bde1a18
  Created wheel for pyeapi: filename=pyeapi-1.0.4-py3-none-any.whl size=94072 sha256=e35baf4371532a676eaa3dd03325efa2be78abd0ebe8e29db643c57922d72e8b
  Stored in dir

In [1]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


# Network Automation lab 1 : Getting started with netmiko

In thus lab, we will:
1. Install the necessary python libraries e.g.
 - Netmiko
 - Requests
 -Napalm
2. Connect to a remote cisco IOS-XE router.
3. Retrieve the running configuration and interface status
4. Push configuration changes {adding a loopback interface}

### Define the device credentials

We use a **Cisco DevNet Always-on sandbox**

This is  a real router that is always reachable over the internet.

 Connecthandler acts as a universal adapter.

 ### Roles of `ConnectHandler`
-Establishes SSH connection to network device and
-Automatically handles the "handshake" (logging in, handles the # or '>' prompts.

**NB** The moment you specify a device_type e.g. cisco_ios, 'Connecthandler' knows how to
- specifically navigate that specific IoS
- enter configuration mode, and
- wait for the deviceto finish processing a command

### `getpass` (The security Gatekeeper)
`getpass` is a standard python module used to keep you credentials safe.

The problem is - when you hard-code your password directly into script (e.g., `password = 'Ry@n123'),` anyone who see your code or has access to your Githib repository can steal it.

**Soln**: `getpass` prompts the user for a password in the terminal/console but hides the characters  as they are typed {it won't show dots or asterisks; it just stays blank}

### How `getpass` and `ConnectHandler` work together

The combination of the two in real-world automation script is both powerful and secure, i.e/,

- `getpass` : safely collects your password from you at runtime { human-to-machine communicatrion}
- `Connecthandler`: Takes the password and uses it to log into the router / switch to run command
{machine-to-machine communication}

In [3]:
from netmiko import ConnectHandler #ConnectHandler - the primary function within netmiko. Different vendors(cisco, Juniper, Huawei, Arista,) use different command-line syntaxes, Connecthandler acts as a universal adapter.
import getpass

#Device details for cisco devNet sandbox

device = {
    'device_type':  'cisco_xe',
    'host': 'sandbox-iosxe-latest-1.cisco.com',
    'username': 'developer',
    'password' : 'Ry@n123', # public since we are not using getpass
    'port': 22,

}

print(f"targeting: {device['host']}")

targeting: sandbox-iosxe-latest-1.cisco.com


## Retrieve information (Show the commands)

Goal:
- Establish a connection and
- pull the interface status

This is the "Read " part of the automation.

In [5]:
try:
  # Establish SSH connection
  print("Connecting to the device...")
  net_connect =  ConnectHandler(**device)

  #Run a show command
  output = net_connect.send_command("show ip interface brief")

  print("\n--- interface Status ---")
  print(output)

  #  Disconnect
  net_connect.disconnect()

except Exception as e:
    print(f"Error connecting to the edevice: {e}")


Connecting to the device...
Error connecting to the edevice: Authentication to device failed.

Common causes of this problem are:
1. Invalid username and password
2. Incorrect SSH-key file
3. Connecting to the wrong device

Device settings: cisco_xe sandbox-iosxe-latest-1.cisco.com:22


Authentication failed.


### Pushing configurations

Automation is particularly spectacular when you want to change settings.

Next, let us create a new virtual interface called Loopback99

In [6]:
# Re-establish connections for the config change

net_connect = ConnectHandler(**device)

config_commands = [
   'interface Loopback99',
   'description configured via python Netmiko',
   'ip address 192.268.99.2 255.255.255.255'
]

print("Sending configuration commands...")
output = net_connect.send_config_set(config_commands)
print(output)

#verify the change
verification = net_connect.send_command("show ip interface brief | include Loopback99")
print("\nVerification")
print(verification)

net_connect.disconnect

NetmikoAuthenticationException: Authentication to device failed.

Common causes of this problem are:
1. Invalid username and password
2. Incorrect SSH-key file
3. Connecting to the wrong device

Device settings: cisco_xe sandbox-iosxe-latest-1.cisco.com:22


Authentication failed.